###Pre-Lab Setup

In [11]:
!pip install spacy
!python -m spacy download en_core_web_sm

import spacy
nlp = spacy.load('en_core_web_sm')
print('spaCy ready!')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 41.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
spaCy ready!


## Task 1.1: Extract Dates

In [12]:
import re
from datetime import datetime

def extract_dates(text):
    """Extract dates in various formats (MM/DD/YYYY, DD-MM-YYYY, Month DD, YYYY)."""
    patterns = [
        r'\d{1,2}/\d{1,2}/\d{4}',
        r'\d{1,2}-\d{1,2}-\d{4}',
        r'\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]* \d{1,2},? \d{4}',
        r'\d{4}-\d{2}-\d{2}'
    ]
    dates = []
    for pattern in patterns:
        matches = re.findall(pattern, text)
        dates.extend(matches)
    return dates

# Test
text = "Invoice date: 03/15/2024. Due: March 30, 2024"
print(f"Dates Found: {extract_dates(text)}")

Dates Found: ['03/15/2024', 'March 30, 2024']


###Task 1.2: Extract Currency Amounts

In [13]:
def extract_amounts(text):
    """Extract currency amounts and convert them to float values."""
    pattern = r'\$?\d{1,3}(?:,\d{3})*(?:\.\d{2})?'
    amounts = re.findall(pattern, text)
    cleaned = [float(amount.replace('$', '').replace(',', '')) for amount in amounts]
    return cleaned

# Test
text = "Total: $1,250.50. Tax: $125.05. Subtotal: 1125.45"
print(f"Amounts Found: {extract_amounts(text)}")

Amounts Found: [1250.5, 125.05, 112.0, 5.45]


###Task 1.3: Extract Invoice Numbers

In [14]:
def extract_invoice_number(text):
    """Identify invoice or order numbers using common prefix patterns."""
    patterns = [r'INV-\d{4}-\d{3}', r'#\d{5,}', r'ORDER-[A-Z0-9]+', r'Invoice (?:Number|#):?\s*([A-Z0-9-]+)']
    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return match.group(1) if match.groups() else match.group(0)
    return None

# Test
text = "Invoice Number: INV-2024-001"
print(f"Invoice ID: {extract_invoice_number(text)}")

Invoice ID: INV-2024-001


# Named Entity Recognition (NER)

## Task 2.1 & 2.2: Extract and Categorize Entities

In [15]:
def extract_entities(text):
    """Categorize text into specific entities like Persons, Organizations, and Locations."""
    doc = nlp(text)
    entities = {'persons': [], 'organizations': [], 'locations': [], 'dates': [], 'money': []}

    for ent in doc.ents:
        if ent.label_ == 'PERSON': entities['persons'].append(ent.text)
        elif ent.label_ == 'ORG': entities['organizations'].append(ent.text)
        elif ent.label_ in ['GPE', 'LOC']: entities['locations'].append(ent.text)
        elif ent.label_ == 'DATE': entities['dates'].append(ent.text)
        elif ent.label_ == 'MONEY': entities['money'].append(ent.text)
    return entities

# Test
sample_text = "Invoice from Acme Corporation, New York. Contact: John Smith. Date: March 15, 2024"
print(extract_entities(sample_text))

{'persons': ['John Smith'], 'organizations': ['Acme Corporation'], 'locations': ['New York'], 'dates': ['March 15, 2024'], 'money': []}


## Task 2.3: Visualize with displaCy

In [16]:
from spacy import displacy

# Create a visual, color-coded representation of entities in the notebook
doc = nlp(sample_text)
displacy.render(doc, style='ent', jupyter=True)

## Complete Extraction Pipeline

In [17]:
# Install Tesseract for OCR
!apt-get install tesseract-ocr
!pip install pytesseract

import json
import pytesseract
from PIL import Image

def process_invoice(image_path):
    """Full pipeline: Perform OCR, apply regex, run NER, and structure as JSON."""
    # Step 1: OCR
    try:
        text = pytesseract.image_to_string(Image.open(image_path))
    except FileNotFoundError:
        return "Please upload 'sample_invoice.jpg' to the Colab file sidebar."

    # Step 2 & 3: Extraction
    invoice_data = {
        'invoice_number': extract_invoice_number(text),
        'dates': extract_dates(text),
        'amounts': extract_amounts(text),
        **extract_entities(text)
    }

    # Step 4: Post-process
    if invoice_data['amounts']: invoice_data['total_amount'] = max(invoice_data['amounts'])
    if invoice_data['dates']: invoice_data['invoice_date'] = invoice_data['dates'][0]

    return invoice_data



Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [22]:
from google.colab import files
import os

uploaded = files.upload()


for filename in uploaded.keys():
    os.rename(filename, 'sample_invoice.jpg')
    print(f"Successfully uploaded and renamed {filename} to 'sample_invoice.jpg'")

Saving Screenshot 2026-05-03 180117.png to Screenshot 2026-05-03 180117.png
Successfully uploaded and renamed Screenshot 2026-05-03 180117.png to 'sample_invoice.jpg'


In [23]:

result = process_invoice('sample_invoice.jpg')

print(json.dumps(result, indent=2))

output_file = 'extracted_data.json'
with open(output_file, 'w') as f:
    json.dump(result, f, indent=2)

print(f'\nSuccess! Results saved to {output_file}')

{
  "invoice_number": null,
  "dates": [],
  "amounts": [
    0.0,
    0.0,
    0.0,
    422.0,
    102.0,
    0.0,
    43.04,
    43.04,
    6.0,
    1.0,
    0.0,
    0.0,
    0.0,
    15.0,
    422.0,
    50.0,
    20.0,
    21.18,
    422.0,
    4.0,
    11.0,
    1.73,
    422.0,
    31.0,
    49.0,
    13.29,
    422.0,
    2.0,
    12.0,
    0.89,
    422.0,
    3.0,
    27.0,
    1.38,
    422.0,
    0.0,
    50.0,
    0.21,
    422.0,
    10.0,
    73.0,
    4.53,
    54.75,
    97.79,
    2.69,
    2.69,
    230.0,
    220.0
  ],
  "persons": [
    "Meter 000000000",
    "Meter 000000000"
  ],
  "organizations": [
    "ABC SUPPLY COMPANY",
    "Subtotal Supplier Services",
    "Energy Efficiency",
    "Subtotal Delivery Services"
  ],
  "locations": [],
  "money": [
    "43.04",
    "43.04",
    "15.00",
    "21.18",
    "13.29",
    "0.89",
    "1.38",
    "0.21",
    "4.53",
    "54.75",
    "97.79",
    "2.69",
    "2.69"
  ],
  "total_amount": 422.0
}

Success! Results sa